# 04 — Collecting C-SPAN abstracts as the summarization reference

**Purpose.** This notebook collects C-SPAN's editorial abstract for every
congressional committee program in its Video Library. Those abstracts are the
reference against which the summaries this pipeline generates are checked: they
are written by people, about the same hearings, with no knowledge of anything
done here, which is what an independent reference has to be.

**Know what the reference is before using it.** Measured over the 16,853
abstracts collected in an earlier pass, the median abstract is **48 words**
against **388** for a summary this pipeline generates — an eight-fold difference,
and a difference in kind:

> **C-SPAN, 36 words.** "The Senate Commerce Committee held a hearing to examine
> supply chain resiliency. Industry and supply chain experts testified on supply
> chain security, the strength of the manufacturing base, private and public
> partnerships, and competition with China."
>
> **Generated, 404 words.** "The congressional hearing focused on addressing the
> US's supply chain challenges and investing in emerging technologies… **1.
> Interagency Coordination**: … **2. Public-Private Partnerships**: …"

C-SPAN writes *what the hearing was about*; the pipeline writes *what was said*.
That supports a **discriminant** test — does the generated summary of hearing X
sit closer to C-SPAN's abstract of X than to any other hearing's — and it does
not support ROUGE or BLEU, where an eight-fold length gap and a genre gap would
be most of whatever the score measured. `06_validate_summaries` builds the
first; anything reporting the second has to say this alongside it.

**Inputs.**

| Source | Endpoint | Contents |
|---|---|---|
| C-SPAN Archives API | `GET /2.0/programs/search` | Lucene query, cursor paging, 20 per page. Returns `id`, `title`, `description`, `date`, `videoLink`. |
| C-SPAN Archives API | `GET /2.0/programs/{id}` | Adds `publicId`, `eventid`, `videoDuration`, `time`. |

The key comes from a `.env` in this directory and is requested from
api@c-spanarchives.org — it is not self-serve.

**Method.**

1. Query the three committee categories separately — `House Committee`,
   `Senate Committee`, `Joint Committee`. There is no combined value; a query
   for "Congressional Committee" returns nothing.
2. Slice by calendar year, so each cursor walk is short and any year can be
   re-collected on its own. Dates need full ISO timestamps:
   `date:[2024-01-01T00:00:00Z TO 2024-12-31T23:59:59Z]`. The shorter forms the
   field appears to invite — `date:2024*`, `date:[2024 TO 2025]` — all return
   500.
3. Follow the cursor to the end of each slice. `lib/cspan.py` raises rather than
   spinning if a cursor stops advancing, which is the one failure a paging walk
   can have that looks like success.
4. Write one JSONL per category-year, then merge to parquet. Re-running skips
   what is already on disk. The abstract, title and date all arrive from the
   search, so nothing else has to be fetched to build the reference set.

**The per-program detail call is off by default, and that is a finding rather
than a shortcut.** `GET /programs/{id}` adds four fields the search does not
return — `publicId`, `eventid`, `videoDuration`, `time` — and the first two were
the reason to make 18,700 extra requests. Neither survived checking:

* `publicId` was supposed to be needed to line this collection up against an
  earlier one. It is not: the earlier collection also carries `videoId`, and the
  search's `id` **is** that number. Joined on it, 4,708 of a 5,130-record sample
  match, so the two collections reconcile without the detail call at all.
* `eventid` was supposed to group the parts of a multi-day hearing, which would
  have resolved the collisions where "Day 1" and "Day 2" both match one printed
  volume. Tested on 60 multi-part groups: **21 share an eventid and 39 do not.**
  "Exploitation of Children over the Internet" Days 1, 2 and 3 carry 142539,
  143097 and 147214. The field identifies a broadcast event, not a hearing.

What `eventid` does do is mark re-airs: "Making the Internet Safe for Children,
Day 1" appears on 2006-06-27 and again on 2006-08-16 under the same 147219. If
duplicate broadcasts ever need removing, that is the field to use, and
`FETCH_DETAIL = True` collects it.

**Outputs.**

| File | Path | Contents |
|---|---|---|
| Search results | `CSPAN_SEARCH / {category}_{year}.jsonl` | One line per program. |
| Program records | `CSPAN_DETAIL / {videoId}.json` | One file per program, only when `FETCH_DETAIL`. |
| Merged table | `CSPAN_RAW / programs.parquet` | One row per program. |

**Notes.**

The site itself is not a route to this data. c-span.org sits behind an AWS WAF
that answers a plain HTTP client with `202` and an empty body after a handful of
requests, and robots.txt disallows the `/ajax` paths its own search page calls.
The API is the supported way in, and it returns the abstract in full rather than
the truncated form the search page shows.

## Setup

`FIRST_YEAR` is 1979, when C-SPAN began broadcasting. The earlier collection
carried records dated as far back as 1948, which are re-aired historical
material rather than contemporaneous coverage; the year loop below reports what
each year actually returns rather than assuming where the archive starts.

In [1]:
import json
import sys
from datetime import datetime, timezone

sys.path.insert(0, "../lib")

import pandas as pd

from cspan import COMMITTEE_CATEGORIES, CSpanClient, CSpanError, load_api_key
from paths import CSPAN_DETAIL, CSPAN_RAW, CSPAN_SEARCH

FIRST_YEAR = 1979
LAST_YEAR = datetime.now(timezone.utc).year

# The detail endpoint adds publicId, eventid, videoDuration and time at a cost
# of one request per program. See the header: the two fields that justified it
# did not survive checking. Set True to collect it anyway — the run is
# resumable, so turning it on later costs only the programs not yet fetched.
FETCH_DETAIL = False

for directory in (CSPAN_SEARCH, CSPAN_DETAIL):
    directory.mkdir(parents=True, exist_ok=True)

client = CSpanClient(api_key=load_api_key(env_file=".env"))

print(f"categories : {COMMITTEE_CATEGORIES}")
print(f"years      : {FIRST_YEAR}-{LAST_YEAR}")
print(f"search  -> {CSPAN_SEARCH}")
print(f"detail  -> {CSPAN_DETAIL}")

categories : ('House Committee', 'Senate Committee', 'Joint Committee')
years      : 1979-2026
search  -> /scratch2/acroh/projects/llm_method/data/raw/cspan/search
detail  -> /scratch2/acroh/projects/llm_method/data/raw/cspan/detail


## Step 1 — Search every category and year

One cursor walk per category-year, written to its own JSONL. A slice already on
disk is skipped, so this notebook is safe to interrupt and re-run: the API is
the slow part and nothing is re-fetched.

Empty years are expected and are written as empty files rather than left
missing, so a later run can tell "collected, nothing there" from "never
collected".

In [2]:
def slice_path(category, year):
    return CSPAN_SEARCH / f"{category.replace(' ', '_').lower()}_{year}.jsonl"


rows = []
for category in COMMITTEE_CATEGORIES:
    per_year = []
    for year in range(FIRST_YEAR, LAST_YEAR + 1):
        path = slice_path(category, year)
        if path.exists():
            programs = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
        else:
            programs = client.search_year(year, category)
            path.write_text("".join(json.dumps(p) + "\n" for p in programs))
        for p in programs:
            rows.append({**p, "category": category, "query_year": year})
        per_year.append((year, len(programs)))

    total = sum(n for _, n in per_year)
    first = next((y for y, n in per_year if n), None)
    print(f"{category:18} {total:>6,} programs, first year with any: {first}")

search = pd.DataFrame(rows)
print(f"\n{len(search):,} rows, {search['id'].nunique():,} distinct programs")
print(f"{client.n_requests:,} API requests, {client.n_retries} retries")

House Committee     8,396 programs, first year with any: 1980
Senate Committee    9,954 programs, first year with any: 1981
Joint Committee       347 programs, first year with any: 1983



18,697 rows, 18,697 distinct programs
0 API requests, 0 retries


## Step 2 — Coverage by year

A program can sit in more than one category, so distinct-program counts are the
ones that matter. The year profile is also the first check on the collection: a
year that suddenly drops to a fraction of its neighbours is a truncated walk
before it is a quiet year in Congress.

In [3]:
by_year = (
    search.pivot_table(index="query_year", columns="category",
                       values="id", aggfunc="nunique", fill_value=0)
    .assign(distinct=search.groupby("query_year")["id"].nunique())
)
print(by_year.to_string())

category    House Committee  Joint Committee  Senate Committee  distinct
query_year                                                              
1980                      1                0                 0         1
1981                      0                0                19        19
1982                      0                0                 7         7
1983                     14                1                19        34
1984                      5                0                13        18
1985                     40                2                47        89
1986                     59                8                66       133
1987                     69               56               127       252
1988                    128               11               140       279
1989                    198                7               189       394
1990                    192               14               222       428
1991                    192               12       

## Step 3 — Optionally, the full record for each program

Skipped unless `FETCH_DETAIL`. One request per program, so this is the long part
of the notebook when it runs — and the header explains why the two fields that
would have paid for it do not.

In [4]:
ids = sorted(search["id"].dropna().astype(int).unique())
missing = [i for i in ids if not (CSPAN_DETAIL / f"{i}.json").exists()] if FETCH_DETAIL else []
if not FETCH_DETAIL:
    print(f"FETCH_DETAIL is off; {len(list(CSPAN_DETAIL.glob('*.json'))):,} detail "
          f"records already on disk are used where present")
else:
    print(f"{len(ids):,} distinct programs, {len(missing):,} still to fetch "
          f"(~{len(missing) / client.rate_per_sec / 60:.0f} min)")

failures = []
for n, video_id in enumerate(missing, 1):
    try:
        record = client.program(video_id)
    except CSpanError as exc:
        failures.append({"id": video_id, "error": str(exc)[:200]})
        continue
    (CSPAN_DETAIL / f"{video_id}.json").write_text(json.dumps(record))
    if n % 2000 == 0:
        print(f"  {n:,}/{len(missing):,}")

print(f"done. {len(failures)} failed")
if failures:
    print(pd.DataFrame(failures).head(10).to_string(index=False))

FETCH_DETAIL is off; 1,331 detail records already on disk are used where present
done. 0 failed


## Step 4 — Merge and persist

Search and detail joined on `id`. Where both carry a field the detail record
wins, since it is the record for that one program rather than a search-result
projection of it.

In [5]:
detail_files = sorted(CSPAN_DETAIL.glob("*.json"))
detail = pd.DataFrame(json.loads(p.read_text()) for p in detail_files) if detail_files \
    else pd.DataFrame(columns=["id"])
print(f"detail records available: {len(detail):,}")

categories = (
    search.groupby("id")["category"]
    .agg(lambda s: "|".join(sorted(set(s))))
    .rename("categories")
)

# The search is the spine: every program has a search row, only some have a
# detail record. Left-joining the other way would drop most of the corpus.
base = search.drop_duplicates("id").drop(columns=["category", "query_year"])
extra = [c for c in detail.columns if c not in base.columns and c != "id"]
programs = (
    base.merge(detail[["id", *extra]], on="id", how="left") if extra else base
).merge(categories, left_on="id", right_index=True, how="left") \
    .assign(word_count=lambda d: d["description"].fillna("").str.split().str.len()) \
    .sort_values("date").reset_index(drop=True)
programs.to_parquet(CSPAN_RAW / "programs.parquet", index=False)

print(f"\nwrote {CSPAN_RAW / 'programs.parquet'}  ({len(programs):,} programs)")
print(f"date span: {programs['date'].min()[:10]} to {programs['date'].max()[:10]}")
have = programs["description"].fillna("").str.strip().ne("")
print(f"with an abstract: {have.sum():,} ({have.mean():.1%})")
print(f"abstract words: median {int(programs.loc[have, 'word_count'].median())}, "
      f"p90 {int(programs.loc[have, 'word_count'].quantile(0.9))}")

detail records available: 1,331



wrote /scratch2/acroh/projects/llm_method/data/raw/cspan/programs.parquet  (18,697 programs)
date span: 1980-10-01 to 2026-08-06
with an abstract: 17,838 (95.4%)
abstract words: median 51, p90 101


## Verification

Four checks, each naming the failure it catches.

1. **One row per program.** A duplicate `id` would mean the merge fanned out, and
   every count taken over this table afterwards would be inflated.
2. **Every searched program has a detail record.** The gap between the two is
   the number of programs the second pass failed on; a non-zero count means
   re-run rather than proceed.
3. **`date` falls inside the year the slice was queried under.** The API filters
   on `date`, so a record outside its own window means the query and the data
   disagree and the year partition is wrong.
4. **Abstract coverage is reported, not asserted.** A program without an
   abstract cannot serve as a reference, so what matters is knowing how many
   there are and in which years — not failing on them.

In [6]:
dupes = int(programs["id"].duplicated().sum())
assert dupes == 0, f"{dupes} duplicate program ids after the merge"
print(f"1) {len(programs):,} rows, one per program")

searched = set(search["id"].dropna().astype(int))
fetched = set(programs["id"].dropna().astype(int))
gap = searched - fetched
assert not gap, f"{len(gap):,} searched programs are missing from the table, e.g. {sorted(gap)[:5]}"
print(f"2) all {len(searched):,} searched programs are in the table")

year_of_slice = search.drop_duplicates("id").set_index("id")["query_year"]
issued = pd.to_datetime(programs["date"], format="mixed", utc=True).dt.year
off = int((issued.values != programs["id"].map(year_of_slice).values).sum())
assert off == 0, f"{off} programs fall outside the year they were queried under"
print("3) every program's date falls inside its query year")

no_abstract = programs.loc[~have]
print(f"4) {len(no_abstract):,} programs have no abstract "
      f"({len(no_abstract) / len(programs):.1%})")
if len(no_abstract):
    print(no_abstract.assign(y=lambda d: d["date"].str[:4])
          .groupby("y").size().rename("n").to_string())

1) 18,697 rows, one per program
2) all 18,697 searched programs are in the table
3) every program's date falls inside its query year
4) 859 programs have no abstract (4.6%)
y
1981      2
1982      1
1983      1
1985      1
1986      1
1987      3
1988      2
1989      8
1990      3
1991      1
2001      2
2002      4
2003     14
2004      5
2005      2
2012      2
2013      9
2014    127
2015    483
2016     95
2017      8
2018      6
2019      1
2020      3
2021      5
2022      5
2023     38
2024     11
2025      4
2026     12
